In [ ]:
import json

import chromadb
from sentence_transformers import SentenceTransformer

articles = json.load(open("../data/processed/civil_code.json"))
by_num = {a["article_number"]: a for a in articles}
rows = [a for a in articles if a["text_ar"].strip()]
assert len({a["article_id"] for a in rows}) == len(rows)

model = SentenceTransformer("intfloat/multilingual-e5-small")
embs = model.encode(
    [f"passage: {a['text_ar']}" for a in rows],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

col = chromadb.EphemeralClient().create_collection("civil_code", metadata={"hnsw:space": "cosine"})
col.add(
    ids=[a["article_id"] for a in rows],
    embeddings=embs.tolist(),
    documents=[a["text_ar"] for a in rows],
    metadatas=[{"article_number": a["article_number"]} for a in rows],
)
print(col.count())

: 

In [ ]:
def search(q: str, k: int = 3) -> list[int]:
    e = model.encode([f"query: {q}"], normalize_embeddings=True)
    r = col.query(query_embeddings=e.tolist(), n_results=k)
    return [m["article_number"] for m in r["metadatas"][0]]

In [ ]:
eval_set = [
    ("ماذا يطبق القاضي إذا لم يوجد نص تشريعي؟", 1),
    ("What does the judge apply if there is no applicable legislative provision?", 1),
    ("هل يجوز لأحد المتعاقدين تعديل العقد بإرادته المنفردة؟", 147),
    ("Can one party unilaterally modify a contract?", 147),
    ("هل يجب تنفيذ العقد بحسن نية؟", 148),
    ("Must a contract be performed in good faith?", 148),
    ("ما التزام من يتسبب بخطئه في ضرر للغير؟", 163),
    ("What is the obligation of a person whose fault causes harm to another?", 163),
    ("ما هو سن الرشد؟", 44),
    ("What is the age of majority?", 44),
]

In [ ]:
for n in sorted({g for _, g in eval_set}):
    print(n, by_num[n]["text_en"][:150], "\n")

In [ ]:
hits = 0
for q, gold in eval_set:
    got = search(q)
    ok = gold in got
    hits += ok
    print("✅" if ok else "❌", gold, got, q)
print("hit@3 =", hits / len(eval_set))

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from groq import Groq

load_dotenv(find_dotenv())
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "qwen/qwen3.6-27b"  # انسخي اسم موديل حالي من console.groq.com/docs/models

SYSTEM = (
    "أنت مساعد قانوني. أجب بالعربية الفصحى وبالاعتماد فقط على المواد المرفقة. "
    "اذكر رقم المادة التي اعتمدت عليها. إذا لم تحتوِ المواد على الإجابة فقل ذلك صراحة."
)


def answer(q: str, k: int = 3) -> str:
    nums = search(q, k)
    ctx = "\n\n".join(f"المادة {n}: {by_num[n]['text_ar']}" for n in nums)
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"المواد:\n{ctx}\n\nالسؤال: {q}"},
        ],
    )
    return resp.choices[0].message.content


print(answer("ما هو سن الرشد؟"))